# Gemini 3 Batch Transcription Pipeline

This Colab orchestrates the transcription of audio segments using Google's **Gemini 3** model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/gemini/gemini_transcribe_audio.ipynb)

### Workflow Overview:
1.  **Job Submission**: Reads a JSONL manifest from GCS and submits asynchronous batch jobs in chunks of 15 segments. It automatically checks for existing transcripts and skips already processed files to avoid duplicate work and optimize API costs (Delta processing).
2.  **Organization**: Results are written directly to GCS.

In [ ]:
!pip install loguru

In [ ]:
import json
import re
import sys
import time

from google import genai
from google.cloud import storage
from google.colab import auth
from loguru import logger

In [ ]:
# @title Define constants and initial logging
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {type:"string"}

GCP_PROJECT_ID = "" # @param {type:"string"}
GCS_BUCKET="" # @param {type:"string"}
GCS_INPUT_DIR="segmented_audio/one_hour_pilot_audio"
EXPERIMENT_NAME = "" # @param {type:"string"}
AUDIO_PREPROCESSING = False # @param {type:"boolean"}

# Handle Preprocessing suffix
if not AUDIO_PREPROCESSING:
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_raw"

# Validation: Ensure required fields are filled to avoid IndexError in downstream GCS calls
assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in the form above."
assert GCS_BUCKET, "GCS_BUCKET must be provided in the form above."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided in the form above."

# Pipeline Control
OVERWRITE_EXISTING = True # @param {type:"boolean"}

# Create a model-specific directory name
MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)
GCS_OUTPUT_BASE = f"transcripts/one_hour_pilot_audio/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
GCP_LOCATION="global"

# Segmentation manifest path
MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/batch_manifest.jsonl"

BATCH_INPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/vertex_batch_input.jsonl"
BATCH_OUTPUT_ROOT = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/batch_results/"

# The consistent final path for consolidated results
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"

SYSTEM_PROMPT = """
ROLE: Literal, mechanical audio-to-text transcription engine.
GOAL: Output exactly and ONLY the words spoken in the audio file.

AUDIO SETTING:
- VHF/UHF emergency radio traffic with heavy static, micro-bursts, and distortion.

STRICT INSTRUCTIONS:
1. ANTI-GUESSING & REJECTION: You are strictly forbidden from guessing. If an audio file is purely static, OR if the speech is too muffled to understand with 100% confidence, do not invent words. You MUST output exactly this token: [UNINTELLIGIBLE]
2. AUDIO LOCALITY: Transcribe ONLY what is audible. Do not continue the speech segment.
3. VERBATIM ONLY: Use digits for numbers (e.g., 10-4, 100, 42).
4. SHORT TRANSMISSIONS: If the transmission is just one or two words (e.g., "copy", "received", "affirm"), output ONLY those words. Do NOT translate meanings.
5. SANITIZATION: No asterisks, markdown, newlines, or roleplay text. Raw text only.

TASK:
Transcribe the audio. Output nothing but the transcript.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"}
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 256,
    "candidate_count": 1,
    "frequency_penalty": 0.3,
    "presence_penalty": 0.0,
    "thinking_config": {
        "include_thoughts": False,
        "thinking_level": "low"
    },
    "top_k": 1,
    "top_p": 0.1
}

logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [ ]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Allow access to the GCS bucket to the VertexAI Service Agent
!gcloud storage buckets add-iam-policy-binding gs://{GCS_BUCKET} \
    --member="serviceAccount:service-$(gcloud projects describe {GCP_PROJECT_ID} --format='value(projectNumber)')@gcp-sa-aiplatform.iam.gserviceaccount.com" \
    --role="roles/storage.objectViewer"

In [ ]:
def _parse_gcs_uri(uri: str) -> tuple[str, str]:
    """Returns (bucket_name, blob_path) from a gs:// URI."""
    without_scheme = uri.removeprefix("gs://")
    bucket, _, blob_path = without_scheme.partition("/")
    return bucket, blob_path


def get_failed_uris(results_uri: str, bucket_name: str) -> list[str]:
    """Identifies URIs that resulted in errors in the final output file."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    path = results_uri.replace(f"gs://{bucket_name}/", "")
    blob = storage_client.bucket(bucket_name).blob(path)
    if not blob.exists():
        return []
    lines = blob.download_as_text().strip().split("\n")
    failed = []
    for line in lines:
        if not line.strip():
            continue
        data = json.loads(line)
        if data.get("status"):
            parts = data["request"]["contents"][0]["parts"]
            uri = next((p.get("file_data", {}).get("file_uri") or p.get("fileData", {}).get("fileUri") for p in parts if "file_uri" in str(p) or "fileUri" in str(p)), "unknown")
            failed.append(uri)
    return failed

def create_batch_request(audio_uri: str) -> dict:
    """Helper to generate a consistent batch request payload."""
    clean_gen_config = GENERATION_CONFIG.copy()
    formatted_sys_instr = {"role": "system", "parts": [{"text": SYSTEM_PROMPT.strip()}]}
    return {
        "request": {
            "contents": [{
                "role": "user",
                "parts": [
                    # Use snake_case for the new GenAI SDK batch endpoint
                    {"file_data": {"file_uri": audio_uri, "mime_type": "audio/flac"}},
                    {"text": "Transcribe this emergency radio communication segment verbatim per the rules above."}
                ]
            }],
            "system_instruction": formatted_sys_instr,
            "generation_config": clean_gen_config,
            "safety_settings": SAFETY_SETTINGS
        }
    }

def create_retry_manifest(failed_uris: list[str], original_manifest_uri: str, retry_manifest_uri: str) -> None:
    """Filters original manifest and wraps in the correct 'request' structure for Vertex Batch."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket_name = original_manifest_uri.replace("gs://", "").split("/")[0]
    path = "/".join(original_manifest_uri.replace("gs://", "").split("/")[1:])
    content = storage_client.bucket(bucket_name).blob(path).download_as_text().strip().split("\n")

    retry_entries = []
    for line in content:
        if not line.strip():
            continue
        entry = json.loads(line)
        if entry["audio_filepath"] in failed_uris:
            batch_entry = create_batch_request(entry["audio_filepath"])
            retry_entries.append(json.dumps(batch_entry))

    if retry_entries:
        out_bucket = retry_manifest_uri.replace("gs://", "").split("/")[0]
        out_path = "/".join(retry_manifest_uri.replace("gs://", "").split("/")[1:])
        storage_client.bucket(out_bucket).blob(out_path).upload_from_string("\n".join(retry_entries))
        logger.info(f"Uploaded retry manifest with {len(retry_entries)} entries.")

def prepare_batch_manifest(input_manifest_uri: str, output_batch_manifest_uri: str, *, overwrite: bool = False, limit: int | None = None) -> str | None:
    """Prepares the JSONL manifest with correct structural requirements for Vertex Batch."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    m_bucket = input_manifest_uri.replace("gs://", "").split("/")[0]
    m_path = "/".join(input_manifest_uri.replace("gs://", "").split("/")[1:])

    try:
        manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
        if not manifest_blob.exists():
            logger.error(f"Manifest not found at {input_manifest_uri}. Check AUDIO_PREPROCESSING settings.")
            return None
        manifest_content = manifest_blob.download_as_text().strip().split("\n")
    except Exception as e:
        logger.error(f"Error reading manifest: {e}")
        return None

    processed_uris = set()
    if not overwrite:
        r_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        r_path = "/".join(CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:])
        results_blob = storage_client.bucket(r_bucket).blob(r_path)
        if results_blob.exists():
            for line in results_blob.download_as_text().strip().split("\n"):
                if line.strip():
                    data = json.loads(line)
                    if not data.get("status"):
                        contents = data["request"].get("contents", [])
                        parts = contents[0].get("parts", []) if isinstance(contents, list) and contents else contents.get("parts", [])
                        uri = next((p.get("file_data", {}).get("file_uri") or p.get("fileData", {}).get("fileUri") for p in parts if "file_uri" in str(p) or "fileUri" in str(p)), None)
                        if uri:
                            processed_uris.add(uri)

    batch_entries = []
    for line in manifest_content:
        if not line.strip():
            continue
        entry = json.loads(line)
        if entry["audio_filepath"] in processed_uris:
            continue

        batch_entry = create_batch_request(entry["audio_filepath"])
        batch_entries.append(json.dumps(batch_entry))
        if limit and len(batch_entries) >= limit:
            logger.info(f"Test limit of {limit} reached.")
            break

    if not batch_entries:
        return None
        out_bucket = output_batch_manifest_uri.replace("gs://", "").split("/")[0]
    out_path = "/".join(output_batch_manifest_uri.replace("gs://", "").split("/")[1:])
    storage_client.bucket(out_bucket).blob(out_path).upload_from_string("\n".join(batch_entries))
    logger.info(f"Prepared {len(batch_entries)} segments for processing at {output_batch_manifest_uri}")
    return output_batch_manifest_uri

def submit_vertex_batch_job(model_id: str, input_uri: str, output_root: str) -> any:
    """Submits a batch prediction job to Vertex AI using the Generative AI SDK."""
    client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=GCP_LOCATION)

    job = client.batches.create(
        model=model_id,
        src=input_uri,
        config={'dest': output_root}
    )
    logger.info(f"Batch job submitted! ID: {job.name}")
    while True:
        job = client.batches.get(name=job.name)
        state = getattr(job.state, "name", str(job.state))
        if state in ["JOB_STATE_SUCCEEDED", "SUCCEEDED", "JOB_STATE_PARTIALLY_SUCCEEDED", "PARTIALLY_SUCCEEDED", "JOB_STATE_FAILED", "FAILED", "JOB_STATE_CANCELLED"]:
            break
        logger.info(f"Current job state: {state}... checking again in 60s")
        time.sleep(60)
    logger.info(f"Job finished with state: {state}")
    return job

def run_automated_retry_pipeline() -> None:
    """Orchestrates checking for failures, creating a retry manifest, and merging results."""
    logger.info("Starting automated error check...")
    failed_uris = get_failed_uris(CONSISTENT_OUTPUT_URI, GCS_BUCKET)
    if not failed_uris:
        logger.info("No failed segments detected. Pipeline complete.")
        return

    logger.info(f"Detected {len(failed_uris)} failures. Creating retry manifest... ")
    RETRY_MANIFEST = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/automated_retry_manifest.jsonl"
    create_retry_manifest(failed_uris, MANIFEST_URI, RETRY_MANIFEST)

    retry_job = submit_vertex_batch_job(MODEL_ID, RETRY_MANIFEST, BATCH_OUTPUT_ROOT)
    if retry_job and getattr(retry_job.state, "name", str(retry_job.state)) in ["JOB_STATE_SUCCEEDED", "SUCCEEDED", "JOB_STATE_PARTIALLY_SUCCEEDED", "PARTIALLY_SUCCEEDED"]:
        consolidate_all_successes(GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI)
    validate_transcription_results(MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET)

def consolidate_all_successes(bucket_name: str, output_base: str, target_uri: str) -> None:
    """Scans all batch result folders and builds a unique map of successful transcriptions."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    prefix = f"{output_base}/batch_results/"
    blobs = bucket.list_blobs(prefix=prefix)

    success_map = {}
    for blob in blobs:
        if "predictions.jsonl" not in blob.name:
            continue

        lines = blob.download_as_text().strip().split("\n")
        for line in lines:
            if not line.strip():
                continue
            data = json.loads(line)
            if not data.get("status"):
                parts = data["request"]["contents"][0]["parts"]
                uri = next((p.get("file_data", {}).get("file_uri") or p.get("fileData", {}).get("fileUri") for p in parts if "file_uri" in str(p) or "fileUri" in str(p)), "unknown")
                if uri not in success_map:
                    success_map[uri] = line

    target_path = target_uri.replace(f"gs://{bucket_name}/", "")
    bucket.blob(target_path).upload_from_string("\n".join(success_map.values()))
    logger.info(f"Consolidation complete. Total unique successful segments: {len(success_map)}")

def validate_transcription_results(manifest_uri: str, results_uri: str, bucket_name: str) -> None:
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    m_bucket = manifest_uri.replace("gs://", "").split("/")[0]
    m_path = "/".join(manifest_uri.replace("gs://", "").split("/")[1:])

    m_blob = storage_client.bucket(m_bucket).blob(m_path)
    if not m_blob.exists():
        logger.warning(f"Validation skipped: Manifest {manifest_uri} does not exist.")
        return

    expected = len([line for line in m_blob.download_as_text().strip().split("\n") if line.strip()])
    r_path = results_uri.replace(f"gs://{bucket_name}/", "")
    results_blob = storage_client.bucket(bucket_name).blob(r_path)
    if results_blob.exists():
        found = len([line for line in results_blob.download_as_text().strip().split("\n") if line.strip()])
        if found == expected:
            logger.info(f"Pipeline Validation: SUCCESS. Expected {expected}, Found {found}.")
        else:
            logger.error(f"Pipeline Validation: FAILURE. Expected {expected}, Found {found}.")

In [ ]:
# @title Main Job Submission
TEST_RUN = False # @param {type:"boolean"}
TEST_LIMIT = 2 # @param {type:"integer"}

actual_batch_input = prepare_batch_manifest(
    input_manifest_uri=MANIFEST_URI,
    output_batch_manifest_uri=BATCH_INPUT_URI,
    overwrite=OVERWRITE_EXISTING,
    limit=TEST_LIMIT if TEST_RUN else None
)

if actual_batch_input:
    logger.info("Submitting main batch job...")
    main_job = submit_vertex_batch_job(
        model_id=MODEL_ID,
        input_uri=actual_batch_input,
        output_root=BATCH_OUTPUT_ROOT
    )
    if main_job:
        state = getattr(main_job.state, "name", str(main_job.state))
        if state in ["JOB_STATE_SUCCEEDED", "SUCCEEDED", "JOB_STATE_PARTIALLY_SUCCEEDED", "PARTIALLY_SUCCEEDED"]:
            logger.info("Main job completed. Consolidating results...")
            consolidate_all_successes(GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI)
            validate_transcription_results(MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET)
        else:
             logger.error(f"Main job failed with state: {state}. Skipping consolidation.")
else:
    logger.info("Skipping main job submission (no new segments to process or manifest missing). Attempting consolidation of previous results...")
    consolidate_all_successes(GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI)
    validate_transcription_results(MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET)

In [ ]:
# @title Automated Retry & Merge Orchestrator
logger.info("Executing automated retry and merge pipeline...")
run_automated_retry_pipeline()